# Kaggle Training Notebook
## Explanation-Supervised Attention — NIH ChestX-ray14

**Run this notebook on Kaggle (free P100/T4 GPU).**

### Setup checklist:
1. Attach the **NIH ChestX-ray14** dataset (read-only) in the Kaggle sidebar
2. Enable **GPU accelerator** (Settings → Accelerator → GPU)
3. Run all cells top-to-bottom

### What this notebook does:
- Phase 2: Train all 3 baselines (ResNet50, DenseNet121, EfficientNet-B0)
- Phase 3: Train the full attention variant
- Phase 4: Run ablations (−L_attn, −L_corr)
- Saves checkpoints + logs to `/kaggle/working/outputs/`

---

In [ ]:
# ============================================================
# Cell 1 — Install dependencies
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'timm', 'albumentations',
                'scikit-learn', 'seaborn', '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# ============================================================
# Cell 2 — Clone / upload project code
# ============================================================
# Option A: Upload the repo as a Kaggle dataset and mount it
# Option B: Use the inline code below (copy src/ into this notebook)
#
# If you uploaded the repo as a dataset named 'dl-project-code':
import sys, os
PROJECT_CODE = '/kaggle/input/dl-project-code'   # adjust to your dataset name
DATASET_PATH = '/kaggle/input/nih-chest-xrays'   # NIH ChestX-ray14 path on Kaggle

if os.path.exists(PROJECT_CODE):
    sys.path.insert(0, PROJECT_CODE)
    print(f'Using project code from: {PROJECT_CODE}')
else:
    print('⚠ Project code not found. Mount dl-project-code dataset in Kaggle sidebar.')

print(f'NIH dataset path: {DATASET_PATH}')
print('Contents:', os.listdir(DATASET_PATH)[:10])

In [ ]:
# ============================================================
# Cell 3 — Config override for Kaggle
# ============================================================
import yaml

cfg = {
    # Paths
    'data_dir':        DATASET_PATH,
    'sample_dir':      os.path.join(DATASET_PATH, 'images'),
    'csv_entry':       'Data_Entry_2017.csv',
    'csv_bbox':        'BBox_List_2017.csv',
    'output_dir':      '/kaggle/working/outputs',
    'checkpoint_dir':  '/kaggle/working/outputs/checkpoints',
    'log_dir':         '/kaggle/working/outputs/logs',
    'figure_dir':      '/kaggle/working/outputs/figures',

    # Mode
    'device':          'cuda',
    'sample_mode':     False,       # use full dataset
    'num_workers':     4,

    # Dataset
    'image_size':      224,
    'num_classes':     14,
    'subset_size':     5000,        # set to None for full dataset

    # Split
    'val_frac':        0.10,
    'test_frac':       0.10,
    'random_seed':     42,

    # Model
    'backbone':           'resnet50',
    'pretrained':         True,
    'attention_resolution': 7,
    'use_channel_attn':   True,

    # Loss
    'lambda1':          1.0,
    'lambda2':          0.5,
    'focal_gamma':      2.0,
    'dice_weight':      1.0,
    'mse_weight':       0.5,
    'sparsity_weight':  0.01,

    # Training
    'epochs':           30,
    'batch_size':       32,
    'lr':               1e-4,
    'weight_decay':     1e-5,
    'scheduler':        'cosine',
    'warmup_epochs':    2,
    'grad_clip':        1.0,
    'mixed_precision':  True,

    # Logging
    'log_interval':         100,
    'val_interval':         1,
    'checkpoint_metric':    'macro_auc',
    'early_stop_patience':  7,
}

os.makedirs(cfg['checkpoint_dir'], exist_ok=True)
os.makedirs(cfg['log_dir'], exist_ok=True)
os.makedirs(cfg['figure_dir'], exist_ok=True)

print('Config ready. Device:', cfg['device'])
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# Cell 4 — Load data, split, EDA summary
# ============================================================
from src.data.splits import (
    load_dataframe, build_balanced_subset,
    patient_level_split, get_class_weights,
    compute_cooccurrence_matrix, CLASS_NAMES
)

df = load_dataframe(cfg['data_dir'])
print(f'Full dataset: {len(df):,} images, {df["patient_id"].nunique():,} patients')

if cfg.get('subset_size'):
    df = build_balanced_subset(df, cfg['subset_size'], cfg['random_seed'])
    print(f'Balanced subset: {len(df):,} images')

train_df, val_df, test_df = patient_level_split(
    df, cfg['val_frac'], cfg['test_frac'], cfg['random_seed']
)

pos_weights = get_class_weights(train_df)
cooc        = compute_cooccurrence_matrix(train_df)
print('Split done. Co-occurrence matrix shape:', cooc.shape)

In [ ]:
# ============================================================
# Cell 5 — Build DataLoaders
# ============================================================
from src.data.dataset import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(
    cfg, train_df, val_df, test_df
)

# Quick shape check
imgs, labels, masks, has_box = next(iter(train_loader))
print(f'Batch shapes: imgs={tuple(imgs.shape)}  labels={tuple(labels.shape)}  '
      f'masks={tuple(masks.shape)}  has_box={has_box.tolist()[:4]}')

In [ ]:
# ============================================================
# Cell 6 — Phase 2: Train ResNet50 Baseline
# ============================================================
from src.train import train

cfg['backbone'] = 'resnet50'
history_resnet_baseline = train(cfg, variant=False)
print('ResNet50 baseline training done.')

In [ ]:
# ============================================================
# Cell 7 — Phase 2: Train DenseNet121 Baseline
# ============================================================
cfg['backbone'] = 'densenet121'
history_dense_baseline = train(cfg, variant=False)
print('DenseNet121 baseline training done.')

In [ ]:
# ============================================================
# Cell 8 — Phase 2: Train EfficientNet-B0 Baseline
# ============================================================
cfg['backbone'] = 'efficientnet_b0'
history_effnet_baseline = train(cfg, variant=False)
print('EfficientNet-B0 baseline training done.')

In [ ]:
# ============================================================
# Cell 9 — Phase 3: Train Attention Variant (ResNet50)
# ============================================================
cfg['backbone'] = 'resnet50'
history_attention = train(cfg, variant=True)
print('ResNet50 attention variant training done.')

In [ ]:
# ============================================================
# Cell 10 — Phase 4: Ablations
# ============================================================
cfg['backbone'] = 'resnet50'

# Ablation (a): no L_attn
print('\n--- Ablation: no L_attn ---')
train(cfg, variant=True, no_lattn=True)

# Ablation (b): no L_corr
print('\n--- Ablation: no L_corr ---')
train(cfg, variant=True, no_lcorr=True)

# Ablation (c): backbone swap — DenseNet121 attention
print('\n--- Ablation: DenseNet121 attention variant ---')
cfg['backbone'] = 'densenet121'
train(cfg, variant=True)

print('All ablations done.')

In [ ]:
# ============================================================
# Cell 11 — Phase 4: Full evaluation + Grad-CAM comparison
# ============================================================
from src.evaluate import full_evaluation

cfg['backbone'] = 'resnet50'
results = full_evaluation(
    cfg,
    variant_ckpt  = f'{cfg["checkpoint_dir"]}/resnet50_attention_best.pt',
    baseline_ckpt = f'{cfg["checkpoint_dir"]}/resnet50_baseline_best.pt',
    backbone      = 'resnet50',
)
print('Evaluation complete.')

In [ ]:
# ============================================================
# Cell 12 — Generate all figures
# ============================================================
import json
from src.plots import (
    plot_training_curves,
    plot_localization_comparison,
    plot_auc_comparison_table,
)

save_dir = cfg['figure_dir']

# Training curves for attention variant
log_path = f"{cfg['log_dir']}/resnet50_attention_log.json"
if os.path.exists(log_path):
    plot_training_curves(log_path, save_dir)

# Localisation comparison
plot_localization_comparison(results, save_dir)

# AUC comparison
plot_auc_comparison_table(results, CLASS_NAMES, save_dir)

print('Figures saved to', save_dir)

In [ ]:
# ============================================================
# Cell 13 — Headline results table
# ============================================================
import pandas as pd

var  = results['variant']
base = results['baseline']
gcam = results['gradcam_loc']

rows = [
    ['Attention Variant (ours)',  var['macro_auc'],
     var['localization']['mean_iou'],
     var['localization']['pointing_game_acc']],
    ['Baseline (Grad-CAM)',       base['macro_auc'],
     gcam['mean_iou'],
     gcam['pointing_game_acc']],
    ['Baseline (no supervision)', base['macro_auc'],
     base['localization']['mean_iou'],
     base['localization']['pointing_game_acc']],
]
tbl = pd.DataFrame(rows, columns=['Model','Macro AUC','Mean IoU','Pointing-Game Acc'])
tbl = tbl.round(4)
display(tbl)
print('\nHeadline table complete.')